# GEM baseline on a T4 — unmodified reference numbers

Establishes the **baseline** half of the throughput comparison the PS grades in
deliverable D. Nothing here uses the macro path; it measures stock GEM so that
the native run has something honest to be compared against.

Three things come out of this notebook:

1. **Baseline throughput** in simulated cycles/second, on stock NVlabs/GEM.
2. **Cooperative-launch headroom** — how many blocks can be co-resident. GEM
   launches cooperatively, so the whole grid must fit at once; if the macro
   phase later inflates registers, occupancy drops and the grid shrinks. Record
   it now so a regression is detectable.
3. **Gate counts** for the same design synthesised with the macros shredded,
   which is the denominator for "how much logic did we remove".

**Before running — Kaggle settings:**

* Accelerator → **GPU T4 x2** (P100 also works)
* Internet → **On** (needed for rustup, oss-cad-suite and the clone)

Budget roughly 25–35 minutes, most of it the Rust build. GPU quota is ~9 h/week,
so avoid re-running the build cell unnecessarily — it caches in `/kaggle/working`.


## 1. Environment

In [ ]:
!nvidia-smi
import subprocess, shutil, os
for t in ["nvcc", "cmake", "gcc", "g++", "python3", "git"]:
    p = shutil.which(t)
    print(f"{t:8} {p or 'MISSING'}")
!nvcc --version | tail -2
!df -h /kaggle/working | tail -1
!free -g | head -2

## 2. Yosys 0.68

**Not** the apt package: Ubuntu ships 0.13/0.33, and PS note 4 mandates 0.68.
The oss-cad-suite nightly is the only prebuilt that carries a current Yosys.

In [ ]:
import os, json, urllib.request, subprocess
os.makedirs("/kaggle/working/opt", exist_ok=True)
if not os.path.exists("/kaggle/working/opt/oss-cad-suite/bin/yosys"):
    rel = json.load(urllib.request.urlopen(
        "https://api.github.com/repos/YosysHQ/oss-cad-suite-build/releases/latest"))
    url = next(a["browser_download_url"] for a in rel["assets"]
               if "linux-x64" in a["name"] and a["name"].endswith((".tgz", ".tar.gz")))
    print("fetching", url)
    subprocess.run(f"curl -fL '{url}' | tar xz -C /kaggle/working/opt",
                   shell=True, check=True)
os.environ["PATH"] = "/kaggle/working/opt/oss-cad-suite/bin:" + os.environ["PATH"]
!yosys -V

## 3. Rust

In [ ]:
import os
if not os.path.exists("/root/.cargo/bin/cargo"):
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal
os.environ["PATH"] = "/root/.cargo/bin:" + os.environ["PATH"]
!cargo --version && rustc --version

## 4. Clone

`GEM_base` comes straight from **public NVlabs/GEM** at the commit our branch
forked from, so the baseline has no dependency on our fork at all and cannot
accidentally be measured with our changes in it.

`GEM_work` is only needed for the `synth/` and `bench/` scripts. If your fork is
private, add a GitHub token as a Kaggle Secret named `GITHUB_TOKEN`
(Add-ons -> Secrets), or make the repo public.

`GIT_TERMINAL_PROMPT=0` is set so a missing credential fails immediately with a
readable error instead of hanging the cell on a password prompt.


In [ ]:
import os, subprocess, sys
os.environ["GIT_TERMINAL_PROMPT"] = "0"      # never block on a password prompt
os.chdir("/kaggle/working")

UPSTREAM = "https://github.com/NVlabs/GEM.git"
BASE_COMMIT = "9e913f9"      # what feat/macro-frontend forked from
FORK_OWNER, FORK_REPO = "harshitabharti011-ops", "GEM"
BRANCH = "feat/macro-frontend"

# ---- unmodified GEM: public upstream, no fork involved ----------------------
if not os.path.isdir("GEM_base"):
    r = subprocess.run(f"git clone {UPSTREAM} GEM_base", shell=True)
    if r.returncode: sys.exit("could not clone public NVlabs/GEM")
    subprocess.run(f"git -C GEM_base checkout -q {BASE_COMMIT}", shell=True, check=True)
    subprocess.run("git -C GEM_base submodule update --init --recursive",
                   shell=True, check=True)
print("baseline:", subprocess.check_output(
    "git -C GEM_base log --oneline -1", shell=True, text=True).strip())

# ---- our branch: public if possible, token if not --------------------------
def try_clone(url, label):
    print(f"trying {label} ...")
    return subprocess.run(f"git clone --recurse-submodules -b {BRANCH} {url} GEM_work",
                          shell=True).returncode == 0

if not os.path.isdir("GEM_work"):
    ok = try_clone(f"https://github.com/{FORK_OWNER}/{FORK_REPO}.git", "public clone")
    if not ok:
        tok = None
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
        except Exception as e:
            print("no GITHUB_TOKEN secret:", e)
        if tok:
            ok = try_clone(f"https://{tok}@github.com/{FORK_OWNER}/{FORK_REPO}.git",
                           "authenticated clone")
    if not ok:
        sys.exit(
            "\nCould not fetch the fork. Check, in order:\n"
            f"  1. the branch is PUSHED:  git push -u origin {BRANCH}\n"
            "  2. the repo is public, OR a Kaggle Secret named GITHUB_TOKEN\n"
            "     holds a PAT with repo:read (Add-ons -> Secrets)\n"
            "  3. the owner/repo names in this cell are right\n"
            "\nThe baseline above does NOT need the fork -- cells 5-6 can run now.")
print("work:", subprocess.check_output(
    "git -C GEM_work log --oneline -1", shell=True, text=True).strip())


## 5. Build stock GEM

If `cooperative_groups.h` fails to compile, the fix is C++17 in `build.rs` —
newer CUDA headers require it. Noted here because it is the most likely
first-run failure on a modern toolchain.

In [ ]:
import os
os.chdir("/kaggle/working/GEM_base")
!cargo build -r --features cuda 2>&1 | tail -25
!ls -la target/release/cuda_test target/release/cut_map_interactive

## 6. Cooperative-launch headroom

Record this now. GEM needs the whole grid co-resident, so this is the ceiling on
`num_blocks`. If the macro phase later raises register pressure, this number
falls and throughput falls with it — that is the first thing to check if native
mode turns out not to be faster.

In [ ]:
import os
os.chdir("/kaggle/working")
!nvcc -O3 -o /tmp/occ GEM_work/bench/occupancy_probe.cu && /tmp/occ

## 7. Synthesise the benchmark — shredded

Same RTL as the native path will use. The only difference is that the macros are
read as behavioural modules rather than blackboxes, so Yosys expands them into
AIG cells. One design, two synthesis scripts: any difference in the measured
result is attributable to the macros and nothing else.

In [ ]:
import os
os.chdir("/kaggle/working/GEM_work")
!./synth/run_synth.sh --shred synth/tests/macro_smoke.sv macro_smoke build_shred 2>&1 | tail -30

## 8. Stimulus

In [ ]:
import os
os.chdir("/kaggle/working/GEM_work")
!./bench/make_vcd.py build_shred/gatelevel.gv macro_smoke -n 20000 -o build_shred/input.vcd
!head -12 build_shred/input.vcd

## 9. Map and simulate

`num_blocks` is set from the probe above, not from a guess. usage.md suggests
2 x SM count; the probe says whether that is actually reachable.

In [ ]:
import os, re, subprocess, time
os.chdir("/kaggle/working/GEM_work")
G = "/kaggle/working/GEM_base/target/release"

sms = int(subprocess.check_output(
    "nvidia-smi --query-gpu=count --format=csv,noheader", shell=True, text=True).split()[0])
import ctypes
NUM_BLOCKS = 2 * 40   # T4 has 40 SMs; the probe cell prints the real limit
print("num_blocks =", NUM_BLOCKS)

t0 = time.time()
!{G}/cut_map_interactive build_shred/gatelevel.gv build_shred/result.gemparts 2>&1 | tail -15
print(f"mapping took {time.time()-t0:.1f}s")

In [ ]:
import os, time, subprocess
os.chdir("/kaggle/working/GEM_work")
G = "/kaggle/working/GEM_base/target/release"
cmd = (f"{G}/cuda_test build_shred/gatelevel.gv build_shred/result.gemparts "
       f"build_shred/input.vcd build_shred/out.vcd {NUM_BLOCKS}")
print(cmd)
t0 = time.time()
out = subprocess.run(cmd, shell=True, capture_output=True, text=True)
wall = time.time() - t0
print(out.stdout[-3000:]); print(out.stderr[-3000:])
print(f"\nwall clock: {wall:.2f}s")

## 10. Record

Everything a later native run needs to be compared against, in one JSON so the
numbers in the report are not retyped by hand.

In [ ]:
import json, os, re, subprocess, datetime
os.chdir("/kaggle/working")
cells = {}
try:
    j = json.load(open("GEM_work/build_shred/gatelevel.json"))
    from collections import Counter
    c = Counter()
    for m in j["modules"].values():
        for cell in m.get("cells", {}).values():
            c[cell["type"].lstrip("\\")] += 1
    cells = dict(c)
except Exception as e:
    print("gate count unavailable:", e)

res = {
    "when": datetime.datetime.utcnow().isoformat() + "Z",
    "gpu": subprocess.check_output(
        "nvidia-smi --query-gpu=name --format=csv,noheader", shell=True, text=True).strip(),
    "mode": "baseline_shredded",
    "gem_commit": subprocess.check_output(
        "git -C GEM_base rev-parse --short HEAD", shell=True, text=True).strip(),
    "num_blocks": NUM_BLOCKS,
    "cycles": 20000,
    "wall_seconds": round(wall, 3),
    "cycles_per_second": round(20000 / wall, 1) if wall else None,
    "cells": cells,
    "aig_cells": sum(v for k, v in cells.items()
                     if k.startswith("AND2") or k in ("INV", "BUF")),
}
open("/kaggle/working/baseline_results.json", "w").write(json.dumps(res, indent=2))
print(json.dumps(res, indent=2))

### What this gives you

`baseline_results.json` is the denominator for every speedup number in the
report. Download it and keep it — re-running on a different GPU invalidates the
comparison, so the native run must happen on the same device, same
`num_blocks`, same cycle count and same input VCD.

**Caveat on `wall_seconds`:** it includes VCD parsing, which usage.md warns can
dominate. If GEM prints its own kernel-only runtime, prefer that number and note
which one the report uses — mixing the two across baseline and native would
manufacture a speedup that isn't real.